In [1]:
import numpy as np
import gzip
import pandas as pd

**Establish subcategory -> category mapping**

In [2]:
data_dir = '/home/ctrader/TF_data/'
metadata_file = data_dir + 'motif_annotations.xlsx'

In [3]:
metadata = pd.read_excel(metadata_file)
metadata

,Cluster_ID,Name,DBD,Seed_motif,Total_width,Consensus_left,Consensus_right,Cluster_size
0,284,AHR,bHLH,AHR_HUMAN.H11MO.0.B,9,2,8,3
1,139,AIRE,not_classified,AIRE_HUMAN.H11MO.0.C,18,2,17,2
2,54,AP1/1,bZIP,ATF3_MOUSE.H11MO.0.A,17,4,12,41
3,53,AP1/2,bZIP,BACH1_HUMAN.H11MO.0.A,19,4,15,12
4,88,ARI5A,ARID,Arid5a_MA0602.1,14,2,10,1
...,...,...,...,...,...,...,...,...
281,222,ZNF784,C2H2,ZNF784_C2H2_1,10,0,10,1
282,127,ZNF85,C2H2,ZNF85_HUMAN.H11MO.0.C,20,2,19,1
283,231,ZNF85,C2H2,ZNF8_HUMAN.H11MO.0.C,22,1,22,1
284,133,ZSCAN3,C2H2,ZSC31_HUMAN.H11MO.0.C,18,1,16,1


In [4]:
mask = (metadata['Name'] == 'SMAD') | (metadata['Name'] == 'ZNF143') | (metadata['Name'] == 'ZNF85')
metadata[mask]

,Cluster_ID,Name,DBD,Seed_motif,Total_width,Consensus_left,Consensus_right,Cluster_size
186,129,SMAD,SMAD,SMAD3_HUMAN.H11MO.0.B,14,2,12,3
187,147,SMAD,SMAD,SMAD2+SMAD3+SMAD4_MA0513.1,13,1,12,4
188,207,SMAD,MAD,SMAD3_MA0795.1,10,1,9,3
235,172,ZNF143,C2H2,ZNF143_C2H2_1,16,1,15,2
236,173,ZNF143,C2H2,IKZF1_HUMAN.H11MO.0.C,24,3,22,7
282,127,ZNF85,C2H2,ZNF85_HUMAN.H11MO.0.C,20,2,19,1
283,231,ZNF85,C2H2,ZNF8_HUMAN.H11MO.0.C,22,1,22,1


In [5]:
x = metadata['Name'].to_numpy()
for m in x:
    if np.count_nonzero(x == m) > 1:
        print(m)

SMAD
SMAD
SMAD
ZNF143
ZNF143
ZNF85
ZNF85


In [6]:
metadata_arr = metadata.to_numpy()

In [7]:
# creating mapping
motif2category = {}
for row in metadata_arr:
    name = row[1]
    dbd = row[2]
    if dbd == 'not_classified':
        dbd = name
    if name not in motif2category:
        motif2category[name] = dbd

In [18]:
dict_out = []
for k, v in zip(list(motif2category.keys()), list(motif2category.values())):
    dict_out.append([k, v])
dict_out = np.array(dict_out)

In [22]:
data_dir = '/home/ctrader/TF_data/'
dict_file = data_dir + 'motif_annotations.name2DBD.CT20230510.tsv'

np.savetxt(dict_file, dict_out, fmt='%s', delimiter='\t')

**Load the raw data**

In [8]:
data_dir = '/home/ctrader/TF_data/'
in_file = data_dir + 'hg38.archetype_motifs.v1.0.bed.gz'

In [9]:
motif_df = pd.read_csv(in_file, compression='gzip', delimiter='\t', usecols=[0, 1, 2, 3], names=['chr', 'start', 'end', 'name'])

In [10]:
motif_df

,chr,start,end,name
0,chr1,10005,10022,KLF/SP/2
1,chr1,10011,10028,KLF/SP/2
2,chr1,10017,10034,KLF/SP/2
3,chr1,10023,10040,KLF/SP/2
4,chr1,10029,10046,KLF/SP/2
...,...,...,...,...
764808786,chrY_KI270740v1_random,37190,37196,SNAI2
764808787,chrY_KI270740v1_random,37190,37197,Ebox/CACCTG
764808788,chrY_KI270740v1_random,37191,37204,NR/11
764808789,chrY_KI270740v1_random,37219,37233,ZNF418


In [15]:
np.unique(list(motif2category.values())).shape

(49,)

In [12]:
motif_df['category'] = np.nan

In [ ]:
for motif in list(motif2category.keys()):
    print(motif)
    mask = motif_df['name'] == motif
    motif_df[mask]['category'] = motif2category[motif]

AHR


In [ ]:
motif_df[mask]

In [ ]:
# # init arr
# out_arr = []

# f = gzip.open(in_file, 'rb')
# current_chrm = ''
# for line in f.readlines():
#     motif_row = line.decode('utf8').strip().split('\t')[:4]
    
#     # track progress
#     if motif_row[0] != current_chrm:
#         print(motif_row[0])
#         current_chrm = motif_row[0]
        
#     # convert motif "Name" to "DBD"
#     if motif_row[3] in motif2category:
#         motif_row[3] = motif2category[motif_row[3]]
#     else:
#         print('error:', motif_row[3])
        
#     out_arr.append(motif_row)
# f.close()

# out_arr = np.array(out_arr)

chr1
chr10


In [ ]:
# Save
data_dir = '/home/ctrader/TF_data/'
out_file = data_dir + 'hg38.archetype_motifs.v1.0.DBD.CT20230509.bed'

np.savetxt(out_file, out_arr, fmt='%s', delimiter='\t')